# Module 10: Challenge Problems & Self-Assessment

**Purpose:** Test your understanding of diffusion models with timed coding exercises, conceptual questions, a bug-finding challenge, and a design discussion.

**How to use this module:**
1. Set a timer for each exercise (time limits noted)
2. Attempt the exercise without looking at the solution
3. After time is up, review the solution and debrief
4. For conceptual questions, write your answer before reading the provided one

**Prerequisites:** Modules 0-9.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, matplotlib.pyplot as plt, math
from tqdm.auto import tqdm
torch.manual_seed(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---

## Exercise 10.1: 2D Point Cloud Diffusion (45 min)

**Set your timer now.**

Implement a complete diffusion model that learns to generate points from a 2D spiral distribution.

**What to implement:**
1. `make_spiral()` -- returns a batch of 2D points
2. Noise schedule tensors (`betas`, `alphas`, `alpha_bar`)
3. Denoiser MLP with sinusoidal timestep embedding
4. Training loop with the DDPM objective
5. DDPM reverse sampling loop
6. Visualization of generated samples vs target

**Checklist:**
- Correct linear schedule with proper `alpha_bar` computation
- Sinusoidal timestep embedding in the denoiser
- Forward process: $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon$
- Correct reverse sampling loop
- Clean, readable code with shape comments

In [ ]:
def make_spiral(n_points: int = 2000, noise: float = 0.3) -> torch.Tensor:
    """Generate a 2D spiral point cloud.

    Args:
        n_points: Number of points to generate.
        noise: Standard deviation of Gaussian noise added to each point.

    Returns:
        Tensor of shape (n_points, 2) with spiral coordinates.
    """
    t = torch.linspace(0, 4 * math.pi, n_points)                 # (n_points,)
    x = t * torch.cos(t) + noise * torch.randn(n_points)         # (n_points,)
    y = t * torch.sin(t) + noise * torch.randn(n_points)         # (n_points,)
    data = torch.stack([x, y], dim=1)                             # (n_points, 2)
    data = (data - data.mean(0)) / data.std(0)                    # normalize to ~N(0,1)
    return data

# Quick check
spiral_data = make_spiral()
plt.figure(figsize=(5, 5))
plt.scatter(spiral_data[:, 0].numpy(), spiral_data[:, 1].numpy(), s=2, alpha=0.5)
plt.title("Training Data: 2D Spiral")
plt.axis("equal")
plt.show()
print(f"spiral_data.shape = {spiral_data.shape}")  # (2000, 2)

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.1: Complete 2D Point Cloud Diffusion

torch.manual_seed(42)

# ---------- Noise schedule ----------
T = 300
betas = torch.linspace(1e-4, 0.02, T).to(device)                    # (T,)
alphas = 1.0 - betas                                                  # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)                             # (T,)


# ---------- Sinusoidal timestep embedding ----------
class SinusoidalEmbedding(nn.Module):
    """Maps scalar timestep to a sinusoidal positional embedding vector."""

    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: (B,) integer timesteps.
        Returns:
            (B, embed_dim) sinusoidal embedding.
        """
        half = self.embed_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device) / half
        )                                                              # (half,)
        args = t[:, None].float() * freqs[None, :]                     # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)   # (B, embed_dim)


# ---------- MLP Denoiser ----------
class MLPDenoiser(nn.Module):
    """Simple MLP that predicts noise given (x_t, t)."""

    def __init__(self, data_dim: int = 2, embed_dim: int = 64, hidden: int = 256):
        super().__init__()
        self.time_embed = SinusoidalEmbedding(embed_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + embed_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 2) noisy data.
            t: (B,) integer timesteps.
        Returns:
            (B, 2) predicted noise.
        """
        t_emb = self.time_embed(t)                       # (B, embed_dim)
        inp = torch.cat([x, t_emb], dim=-1)              # (B, 2 + embed_dim)
        return self.net(inp)                              # (B, 2)


# ---------- Training ----------
data = make_spiral(n_points=2000).to(device)              # (2000, 2)
model = MLPDenoiser().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for step in tqdm(range(5000), desc="Training"):
    # Sample batch from data
    idx = torch.randint(0, len(data), (256,))
    x_0 = data[idx]                                       # (256, 2)

    # Sample random timesteps
    t = torch.randint(0, T, (256,), device=device)        # (256,)

    # Sample noise
    eps = torch.randn_like(x_0)                           # (256, 2)

    # Forward process: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
    ab_t = alpha_bars[t][:, None]                         # (256, 1)
    x_t = torch.sqrt(ab_t) * x_0 + torch.sqrt(1 - ab_t) * eps  # (256, 2)

    # Predict noise
    eps_pred = model(x_t, t)                              # (256, 2)

    # MSE loss
    loss = F.mse_loss(eps_pred, eps)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.3)
plt.plot(np.convolve(losses, np.ones(100)/100, mode='valid'), color='red')
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.show()


# ---------- DDPM Sampling ----------
@torch.no_grad()
def ddpm_sample(model: nn.Module, n_samples: int = 2000) -> torch.Tensor:
    """Generate samples via the full DDPM reverse process.

    Args:
        model: Trained noise-prediction network.
        n_samples: Number of points to generate.

    Returns:
        (n_samples, 2) generated data points.
    """
    model.eval()
    x = torch.randn(n_samples, 2, device=device)            # (n_samples, 2)

    for t_val in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)  # (n_samples,)
        eps_pred = model(x, t_batch)                         # (n_samples, 2)

        beta_t = betas[t_val]
        alpha_t = alphas[t_val]
        alpha_bar_t = alpha_bars[t_val]

        # DDPM mean
        x = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )                                                    # (n_samples, 2)

        # Add noise for all steps except t=0
        if t_val > 0:
            z = torch.randn_like(x)                          # (n_samples, 2)
            x = x + torch.sqrt(beta_t) * z

    model.train()
    return x


samples = ddpm_sample(model)                                 # (2000, 2)

# ---------- Visualization ----------
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(data[:, 0].cpu().numpy(), data[:, 1].cpu().numpy(), s=2, alpha=0.5)
axes[0].set_title("Training Data")
axes[0].set_aspect("equal")
axes[1].scatter(samples[:, 0].cpu().numpy(), samples[:, 1].cpu().numpy(), s=2, alpha=0.5, color="orange")
axes[1].set_title("Generated Samples (DDPM)")
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()

### Exercise 10.1: Solution Debrief

**Self-assessment checklist:**
- Did you get the noise schedule right? (`betas`, `alphas`, `alphas_cumprod`)
- Forward process uses $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1 - \bar\alpha_t} \epsilon$?
- Denoiser includes sinusoidal timestep embeddings?
- Sampling loop applies the DDPM reverse step correctly?
- Noise $z \sim \mathcal{N}(0, I)$ added at every step **except** $t=0$?
- Generated points visually match the target spiral?

**Common mistakes:**
- Using `alphas` instead of `alphas_cumprod` in the forward process
- Missing the $\sqrt{\cdot}$ on both terms
- Not conditioning on timestep in the denoiser
- Adding noise at $t=0$ during sampling

---

## Exercise 10.2: DDIM Sampling (20 min)

**Set your timer now.**

Given a trained DDPM noise predictor, implement DDIM deterministic sampling with a reduced step count.

**You are given:**
- A trained model `model(x_t, t)` that predicts noise $\hat{\epsilon}$
- Schedule tensors `alphas_cumprod` of shape `(T,)`
- A subsequence of timesteps (e.g., every 50th step)

**What to implement:**
- The DDIM update step (deterministic, $\sigma = 0$)
- Sampling using a timestep subsequence for speedup

**Key formula:**

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \hat{x}_0 + \sqrt{1 - \bar\alpha_{t-1}} \cdot \hat\epsilon$$

where $\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t} \cdot \hat\epsilon}{\sqrt{\bar\alpha_t}}$

In [ ]:
# ---------- Compact class-conditional UNet for MNIST ----------
# This cell defines and trains the model so that Exercise 10.2 has a working checkpoint.

torch.manual_seed(42)

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --- Data ---
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
loader = DataLoader(mnist, batch_size=128, shuffle=True, drop_last=True)

# --- Schedule (reuse T=300) ---
T_mnist = 300
betas_m = torch.linspace(1e-4, 0.02, T_mnist).to(device)       # (T,)
alphas_m = 1.0 - betas_m                                        # (T,)
alpha_bars_m = torch.cumprod(alphas_m, dim=0)                    # (T,)

NUM_CLASSES = 10
NULL_CLASS = NUM_CLASSES  # label index for unconditional (dropout)


class ResBlock(nn.Module):
    """Residual block with timestep and class conditioning."""

    def __init__(self, ch: int, emb_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, ch)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, ch, H, W)
            emb: (B, emb_dim) combined time+class embedding
        Returns:
            (B, ch, H, W)
        """
        h = F.silu(self.norm1(x))                                # (B, ch, H, W)
        h = self.conv1(h)                                        # (B, ch, H, W)
        h = h + self.emb_proj(emb)[:, :, None, None]             # broadcast (B, ch, 1, 1)
        h = F.silu(self.norm2(h))                                # (B, ch, H, W)
        h = self.conv2(h)                                        # (B, ch, H, W)
        return x + h                                             # residual


class SmallCondUNet(nn.Module):
    """Minimal class-conditional UNet for 28x28 MNIST images."""

    def __init__(self, in_ch: int = 1, base_ch: int = 32, emb_dim: int = 64):
        super().__init__()
        self.emb_dim = emb_dim
        # Embeddings
        self.time_mlp = nn.Sequential(
            nn.Linear(emb_dim, emb_dim * 2), nn.SiLU(), nn.Linear(emb_dim * 2, emb_dim)
        )
        self.class_emb = nn.Embedding(NUM_CLASSES + 1, emb_dim)  # +1 for null class

        # Encoder
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)          # 28x28
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)  # 14x14
        self.rb1 = ResBlock(base_ch * 2, emb_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1)  # 7x7
        self.rb2 = ResBlock(base_ch * 4, emb_dim)

        # Bottleneck
        self.mid = ResBlock(base_ch * 4, emb_dim)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, stride=2, padding=1)  # 14x14
        self.rb3 = ResBlock(base_ch * 4, emb_dim)  # skip cat doubles channels
        self.up1 = nn.ConvTranspose2d(base_ch * 4, base_ch, 4, stride=2, padding=1)  # 28x28
        self.rb4 = ResBlock(base_ch * 2, emb_dim)

        self.out_conv = nn.Conv2d(base_ch * 2, in_ch, 3, padding=1)

    def _sinusoidal_emb(self, t: torch.Tensor) -> torch.Tensor:
        half = self.emb_dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None, :]             # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, emb_dim)

    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 1, 28, 28) noisy image in [-1, 1].
            t: (B,) integer timesteps.
            y: (B,) class labels (0-9) or NULL_CLASS for unconditional.
        Returns:
            (B, 1, 28, 28) predicted noise.
        """
        emb = self.time_mlp(self._sinusoidal_emb(t)) + self.class_emb(y)  # (B, emb_dim)

        h1 = self.in_conv(x)                    # (B, 32, 28, 28)
        h2 = self.rb1(self.down1(h1), emb)      # (B, 64, 14, 14)
        h3 = self.rb2(self.down2(h2), emb)      # (B, 128, 7, 7)
        h = self.mid(h3, emb)                   # (B, 128, 7, 7)
        h = self.up2(h)                         # (B, 64, 14, 14)
        h = self.rb3(torch.cat([h, h2], 1), emb)  # (B, 128, 14, 14) -> ResBlock(128) -> (B, 128, 14, 14)
        h = self.up1(h)                         # (B, 32, 28, 28)
        h = self.rb4(torch.cat([h, h1], 1), emb)  # (B, 64, 28, 28) -> ResBlock(64) -> (B, 64, 28, 28)
        return self.out_conv(h)                 # (B, 1, 28, 28)


cond_model = SmallCondUNet().to(device)
opt_m = torch.optim.Adam(cond_model.parameters(), lr=1e-3)
p_uncond = 0.1  # 10% chance of dropping class label for CFG training

step = 0
cond_model.train()
for epoch in range(2):  # ~2000 steps with batch 128
    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch}"):
        imgs = imgs.to(device)                                   # (B, 1, 28, 28) in [-1,1]
        labels = labels.to(device)                               # (B,)

        # Random class dropout for CFG
        mask = torch.rand(len(labels), device=device) < p_uncond
        labels = torch.where(mask, torch.full_like(labels, NULL_CLASS), labels)  # (B,)

        t = torch.randint(0, T_mnist, (len(imgs),), device=device)  # (B,)
        eps = torch.randn_like(imgs)                             # (B, 1, 28, 28)
        ab = alpha_bars_m[t][:, None, None, None]                # (B, 1, 1, 1)
        x_t = torch.sqrt(ab) * imgs + torch.sqrt(1 - ab) * eps  # (B, 1, 28, 28)

        eps_pred = cond_model(x_t, t, labels)                    # (B, 1, 28, 28)
        loss = F.mse_loss(eps_pred, eps)

        opt_m.zero_grad()
        loss.backward()
        opt_m.step()
        step += 1
        if step >= 2000:
            break
    if step >= 2000:
        break

print(f"Trained {step} steps. Final loss: {loss.item():.4f}")

In [ ]:
# Starter code -- implement ddim_sample_cfg

def ddim_sample_cfg(
    model: nn.Module,
    num_steps: int = 50,
    guidance_scale: float = 2.0,
    class_label: int = 3,
    n_samples: int = 16,
) -> torch.Tensor:
    """Generate MNIST images using DDIM sampling with classifier-free guidance.

    Args:
        model: Trained class-conditional noise predictor.
        num_steps: Number of DDIM sub-steps (< T for acceleration).
        guidance_scale: CFG weight (w). eps_guided = eps_uncond + w * (eps_cond - eps_uncond).
        class_label: Target digit class (0-9).
        n_samples: Number of images to generate.

    Returns:
        (n_samples, 1, 28, 28) generated images in [-1, 1].
    """
    # YOUR CODE HERE
    # ===================== YOUR CODE HERE =====================
    pass  # Replace this with your implementation
    # ====================== END YOUR CODE ======================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.2: DDIM Sampling with CFG

@torch.no_grad()
def ddim_sample_cfg(
    model: nn.Module,
    num_steps: int = 50,
    guidance_scale: float = 2.0,
    class_label: int = 3,
    n_samples: int = 16,
) -> torch.Tensor:
    """Generate MNIST images using DDIM sampling with classifier-free guidance.

    Args:
        model: Trained class-conditional noise predictor.
        num_steps: Number of DDIM sub-steps (< T for acceleration).
        guidance_scale: CFG weight (w). eps_guided = eps_uncond + w * (eps_cond - eps_uncond).
        class_label: Target digit class (0-9).
        n_samples: Number of images to generate.

    Returns:
        (n_samples, 1, 28, 28) generated images clamped to [-1, 1].
    """
    model.eval()

    # Timestep sub-selection: uniform spacing from T-1 down to 0
    timesteps = torch.linspace(T_mnist - 1, 0, num_steps + 1).long().to(device)  # (num_steps+1,)

    x = torch.randn(n_samples, 1, 28, 28, device=device)        # (n, 1, 28, 28)

    for i in range(num_steps):
        t_cur = timesteps[i]       # current timestep
        t_next = timesteps[i + 1]  # next timestep (closer to 0)

        t_batch = t_cur.expand(n_samples)                        # (n,)

        # --- CFG: batch conditional + unconditional together ---
        x_double = torch.cat([x, x], dim=0)                     # (2n, 1, 28, 28)
        t_double = torch.cat([t_batch, t_batch], dim=0)          # (2n,)
        y_cond = torch.full((n_samples,), class_label, device=device, dtype=torch.long)
        y_uncond = torch.full((n_samples,), NULL_CLASS, device=device, dtype=torch.long)
        y_double = torch.cat([y_cond, y_uncond], dim=0)          # (2n,)

        eps_double = model(x_double, t_double, y_double)         # (2n, 1, 28, 28)
        eps_cond, eps_uncond = eps_double.chunk(2, dim=0)        # each (n, 1, 28, 28)

        # Guided noise prediction
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)  # (n, 1, 28, 28)

        # --- DDIM update ---
        alpha_bar_t = alpha_bars_m[t_cur]
        alpha_bar_next = alpha_bars_m[t_next] if t_next > 0 else torch.tensor(1.0, device=device)

        # Predicted x_0
        predicted_x0 = (x - torch.sqrt(1 - alpha_bar_t) * eps_guided) / torch.sqrt(alpha_bar_t)
        predicted_x0 = predicted_x0.clamp(-1, 1)                # (n, 1, 28, 28)

        # Direction pointing to x_t
        direction = torch.sqrt(1 - alpha_bar_next) * eps_guided  # (n, 1, 28, 28)

        # DDIM deterministic step (eta=0)
        x = torch.sqrt(alpha_bar_next) * predicted_x0 + direction  # (n, 1, 28, 28)

    model.train()
    return x.clamp(-1, 1)


# --- Generate grids at different guidance scales ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, w in zip(axes, [1.0, 2.0, 4.0, 8.0]):
    torch.manual_seed(42)
    imgs = ddim_sample_cfg(cond_model, num_steps=50, guidance_scale=w, class_label=3, n_samples=16)
    # Make a 4x4 grid
    grid = imgs.cpu().view(4, 4, 28, 28)                        # (4, 4, 28, 28)
    grid = grid.permute(0, 2, 1, 3).reshape(4 * 28, 4 * 28)     # (112, 112)
    ax.imshow(grid.numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"w = {w}")
    ax.axis("off")

plt.suptitle("DDIM + CFG: Digit 3 at Various Guidance Scales", fontsize=14)
plt.tight_layout()
plt.show()

### Exercise 10.2: Solution Debrief

**Self-assessment checklist:**
- Correctly computed $\hat{x}_0$ from the noise prediction?
- Used the right $\bar\alpha$ values for both current and previous timesteps?
- Iterated over the subsequence in reverse order?
- Output is deterministic (no random noise added)?

**Key insight:** DDIM works because the model already learned the noise distribution during DDPM training. DDIM just changes how we use those predictions -- a deterministic ODE step instead of a stochastic SDE step.

---

## Exercise 10.3: Classifier-Free Guidance (25 min)

**Set your timer now.**

Modify a class-conditional diffusion model to support classifier-free guidance. You need to implement two things:

1. **Training:** Random label dropout -- replace labels with a null token with probability `p_uncond`
2. **Sampling:** The CFG formula using both conditional and unconditional predictions

**Key formula:**

$$\tilde{\epsilon} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

where $s$ is the guidance scale.

**Shapes to watch:**
- Concatenate `[x_t, x_t]` along dim 0 to get `(2B, C, H, W)`
- Split conditional/unconditional predictions with `.chunk(2)`

In [ ]:
# Starter code -- Exercise 10.3
# Model and data are already available from Exercise 10.2:
#   - SmallCondUNet class is defined
#   - MNIST DataLoader `loader` is ready
#   - device is set

# Create a fresh model for this exercise
# ========================= YOUR CODE HERE =========================

torch.manual_seed(42)
model_ex3 = SmallCondUNet().to(device)
optimizer_ex3 = torch.optim.Adam(model_ex3.parameters(), lr=1e-3)

# YOUR CODE HERE:
# 1. Define noise schedule (T, betas, alphas, alpha_bars)
# 2. Write train_step(model, optimizer, x_batch, y_batch) -> float
# 3. Run 500 training steps, collect losses
# 4. Plot loss curve

# ========================== END YOUR CODE ==========================

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Exercise 10.3: Training Step from Scratch

torch.manual_seed(42)

# ---------- 1. Noise schedule ----------
T_ex3 = 300
betas_ex3 = torch.linspace(1e-4, 0.02, T_ex3).to(device)          # (T,)
alphas_ex3 = 1.0 - betas_ex3                                        # (T,)
alpha_bars_ex3 = torch.cumprod(alphas_ex3, dim=0)                    # (T,)

model_ex3 = SmallCondUNet().to(device)
optimizer_ex3 = torch.optim.Adam(model_ex3.parameters(), lr=1e-3)
p_uncond_ex3 = 0.1


# ---------- 2. train_step ----------
def train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    x_batch: torch.Tensor,
    y_batch: torch.Tensor,
) -> float:
    """Perform one diffusion training step.

    Args:
        model: Noise prediction network (class-conditional).
        optimizer: Optimizer for model parameters.
        x_batch: (B, 1, 28, 28) clean images in [-1, 1].
        y_batch: (B,) class labels.

    Returns:
        Scalar MSE loss value.
    """
    B = x_batch.shape[0]

    # Class dropout for CFG
    mask = torch.rand(B, device=device) < p_uncond_ex3
    y_batch = torch.where(mask, torch.full_like(y_batch, NULL_CLASS), y_batch)  # (B,)

    # Sample random timesteps
    t = torch.randint(0, T_ex3, (B,), device=device)                # (B,)

    # Sample noise
    eps = torch.randn_like(x_batch)                                  # (B, 1, 28, 28)

    # Compute x_t via forward process
    ab = alpha_bars_ex3[t][:, None, None, None]                      # (B, 1, 1, 1)
    x_t = torch.sqrt(ab) * x_batch + torch.sqrt(1 - ab) * eps       # (B, 1, 28, 28)

    # Predict noise
    eps_pred = model(x_t, t, y_batch)                                # (B, 1, 28, 28)

    # MSE loss
    loss = F.mse_loss(eps_pred, eps)

    # Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


# ---------- 3. Training loop ----------
model_ex3.train()
losses_ex3 = []
data_iter = iter(loader)
for step in tqdm(range(500), desc="Training (Ex 10.3)"):
    try:
        imgs, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        imgs, labels = next(data_iter)

    imgs = imgs.to(device)      # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)

    loss_val = train_step(model_ex3, optimizer_ex3, imgs, labels)
    losses_ex3.append(loss_val)

# ---------- 4. Plot loss curve ----------
plt.figure(figsize=(8, 3))
plt.plot(losses_ex3, alpha=0.4, label="Raw loss")
window = 50
smoothed = np.convolve(losses_ex3, np.ones(window) / window, mode="valid")
plt.plot(range(window - 1, len(losses_ex3)), smoothed, color="red", label=f"Smoothed ({window}-step)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Exercise 10.3: Training Loss Curve (500 steps)")
plt.legend()
plt.show()

### Exercise 10.3: Solution Debrief

**Self-assessment checklist:**
- Used `num_classes` (not `num_classes + 1`) as the null token index?
- Dropped labels with the correct probability during training?
- Batched conditional and unconditional forward passes together?
- CFG formula matches $\tilde{\epsilon} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$?
- Generated with $s > 1$ to see the guidance effect?

**Key insight:** The only training change is a single line that randomly replaces labels with the null token. The model implicitly learns both conditional and unconditional denoising from the same training run.

---

## Exercise 10.4: Conceptual Questions

For each question, **write your answer** before reading the provided one. Being able to explain these concepts clearly is the best test of understanding.

---

### Core Concepts

#### Q1: Walk me through the forward diffusion process.

*Write your answer before reading below.*

**Answer (Q1)**

The forward process gradually adds Gaussian noise over T timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t;\; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I)$$

The key insight is the **closed-form jump** to any timestep $t$:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\bar\alpha_t = \prod_{s=1}^t (1 - \beta_s)$.

- At $t=0$: clean data
- At $t=T$: approximately pure Gaussian noise
- Signal attenuates by $\sqrt{\bar\alpha_t}$, noise scales by $\sqrt{1 - \bar\alpha_t}$
- SNR decreases monotonically

**Common mistake:** Confusing $\alpha_t$ (single-step) with $\bar\alpha_t$ (cumulative product).

#### Q2: Why do we predict noise instead of the clean image?

*Write your answer before reading below.*

**Answer (Q2)**

Predicting noise ($\epsilon$-prediction) works better empirically for several reasons:

1. **Uniform difficulty across timesteps.** The target $\epsilon \sim \mathcal{N}(0, I)$ has the same distribution regardless of $t$. Predicting $x_0$ directly would be trivial at low $t$ and impossible at high $t$, creating an imbalanced loss landscape.

2. **Score matching connection.** Predicting $\epsilon$ is equivalent to estimating the score $\nabla_{x_t} \log q(x_t)$, linking to deep theoretical foundations.

3. **Simplified loss.** The objective $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$ drops timestep-dependent weighting and trains more stably.

All three parameterizations ($\epsilon$, $x_0$, $v$) are mathematically interconvertible -- the choice affects training dynamics, not expressiveness.

**Common mistake:** Claiming noise prediction is "theoretically better." It is *empirically* better for standard diffusion; $x_0$-prediction can be superior in other settings.

#### Q3: Explain the reparameterization trick and why it matters for diffusion.

*Write your answer before reading below.*

**Answer (Q3)**

The reparameterization trick rewrites a sample from a parameterized distribution as a deterministic function of parameters plus independent noise:

$$x = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, 1)$$

This makes the operation differentiable w.r.t. $\mu$ and $\sigma$ (the randomness is isolated in $\epsilon$).

In diffusion, it appears in two places:
- **Forward process:** $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon$ is the reparameterized form of $q(x_t|x_0)$
- **Training:** We sample $\epsilon$ independently, construct $x_t$ deterministically, and backpropagate through the model

**Key point:** Without reparameterization, we could not backpropagate through the sampling step. This is what makes the entire training pipeline differentiable.

**Common mistake:** Thinking this is specific to diffusion -- it is a general technique from VAEs used whenever you need gradients through sampling.

#### Q4: Derive or explain the simplified DDPM loss.

*Write your answer before reading below.*

**Answer (Q4)**

Starting from the variational lower bound (VLB) on $\log p(x_0)$:
- It decomposes into T KL-divergence terms, one per timestep
- Each compares the true posterior $q(x_{t-1}|x_t, x_0)$ to the learned reverse $p_\theta(x_{t-1}|x_t)$
- With Gaussian assumptions and fixed variance, each KL becomes an MSE between means

Substituting the $\epsilon$-parameterization gives:

$$L_t = \frac{\beta_t^2}{2\sigma_t^2 \alpha_t (1 - \bar\alpha_t)} \|\epsilon - \epsilon_\theta(x_t, t)\|^2$$

The **simplified loss** drops the timestep-dependent coefficient:

$$L_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

Ho et al. (2020) found this unweighted version trains better. It effectively upweights large-$t$ terms, encouraging the model to denoise from highly corrupted inputs.

**Common mistake:** Claiming the simplified loss IS the variational bound. It is a reweighted version that produces better samples at the cost of a looser bound.

#### Q5: Compare DDPM and DDIM sampling.

*Write your answer before reading below.*

**Answer (Q5)**

| Property | DDPM | DDIM |
|---|---|---|
| **Stochasticity** | Stochastic (adds noise each step) | Deterministic (eta=0) or controllable |
| **Steps required** | All T steps (e.g., 1000) | Any subset (e.g., 50 steps) |
| **Same model?** | Yes | Yes -- no retraining needed |
| **Latent interpolation** | Not meaningful (stochastic) | Meaningful (deterministic mapping) |
| **Few-step quality** | Degrades significantly | Maintains quality |

DDIM defines a non-Markovian process with the same marginals $q(x_t|x_0)$. The update:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \cdot \hat{x}_0 + \sqrt{1-\bar\alpha_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

The `eta` parameter controls stochasticity: eta=0 is deterministic, eta=1 recovers DDPM.

**Common mistake:** Thinking DDIM requires different training. Only the inference loop changes.

---

### Architecture

#### Q6: Why use GroupNorm instead of BatchNorm in diffusion UNets?

*Write your answer before reading below.*

**Answer (Q6)**

GroupNorm normalizes within groups of channels **per sample**, while BatchNorm normalizes across the batch. GroupNorm is preferred because:

1. **Batch independence.** During sampling, batch size is often 1. BatchNorm statistics are meaningless with B=1; GroupNorm works identically regardless of batch size.

2. **Heterogeneous noise levels.** Each sample in a training batch has a different timestep $t$. BatchNorm would compute statistics across wildly different noise levels, mixing signals inappropriately.

3. **No train/eval gap.** GroupNorm avoids BatchNorm's running statistics discrepancy, which is problematic for the long training runs typical of diffusion.

**Common mistake:** Saying GroupNorm is always better. BatchNorm is fine for homogeneous batches (e.g., classification). The issue is specific to diffusion's per-sample timestep conditioning.

---

### Guidance

#### Q7: Explain classifier-free guidance (CFG).

*Write your answer before reading below.*

**Answer (Q7)**

CFG improves conditional sample quality without a separate classifier by training one model for both conditional and unconditional generation.

**Training:** With probability $p$ (typically 10-20%), replace the conditioning signal with a null token. The model learns both modes from the same training run.

**Inference:** At each step, run two forward passes:
- **Conditional:** $\epsilon_\theta(x_t, t, c)$
- **Unconditional:** $\epsilon_\theta(x_t, t, \varnothing)$

Combine: $\tilde{\epsilon} = \epsilon_\text{uncond} + w \cdot (\epsilon_\text{cond} - \epsilon_\text{uncond})$

The difference $(\epsilon_\text{cond} - \epsilon_\text{uncond})$ captures "what the condition adds." Scaling by $w > 1$ amplifies this effect.

**Common mistake:** Setting $w = 1$ and expecting a guidance effect. At $w = 1$, this reduces to plain conditional sampling.

#### Q8: What happens as you increase the guidance scale? What are the tradeoffs?

*Write your answer before reading below.*

**Answer (Q8)**

| Guidance Scale (w) | Fidelity | Diversity | Artifacts |
|---|---|---|---|
| w = 1 | Baseline | High | None |
| w = 2-4 | Good | Moderate | Minimal |
| w = 7-10 | High (typical) | Low | Some saturation |
| w > 15 | Very high | Very low | Oversaturation, distortion |

The core tradeoff is **fidelity vs. diversity** (analogous to precision/recall in GANs, or temperature in LLMs):
- **Low w:** Diverse but weakly conditioned
- **High w:** Sharp and on-target but collapsed toward prototypical examples

At very high $w$, the guided $\tilde{\epsilon}$ has magnitudes far beyond training, pushing into out-of-distribution regions. Mitigations include dynamic thresholding (Imagen) and rescaled CFG.

**Common mistake:** Thinking higher guidance is always better. There is a sweet spot (typically 7-12 for text-to-image).

#### Q9: How do negative prompts work in diffusion models?

*Write your answer before reading below.*

**Answer (Q9)**

Negative prompts modify the CFG formula by replacing the null/empty embedding with an embedding of what you want to *avoid*:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, c_\text{neg}) + w \cdot (\epsilon_\theta(x_t, t, c_\text{pos}) - \epsilon_\theta(x_t, t, c_\text{neg}))$$

The guidance direction becomes "move away from the negative prompt, toward the positive prompt."

**Examples:**
- Positive: "a photo of a cat", Negative: "blurry, low quality" -- produces sharper images
- Positive: "landscape painting", Negative: "people, text, watermark" -- removes unwanted elements

No architectural changes or retraining needed -- it is purely an inference-time swap of the unconditional embedding.

**Common mistake:** Thinking negative prompts require a separate model or loss function.

In [ ]:
# BUGGY CODE -- Find all 10 bugs!
# This code is intentionally broken. Do NOT run this cell.
# Read it carefully and identify the bugs.

buggy_code = """
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = alphas

class BuggyDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 2),
        )

    def forward(self, x, t):
      
        t_input = t.unsqueeze(-1).float()  # just a scalar
        return self.net(torch.cat([x, t_input], dim=-1))

# --- Training ---
model = BuggyDenoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
data = torch.randn(1000, 2)  # assume some training data

for step in range(1000):
    idx = torch.randint(0, len(data), (64,))
    x_0 = data[idx]

    t = torch.randint(0, T, (64,))
    eps = torch.randn_like(x_0)

    # --- Bug 3: Missing sqrt ---
    ab_t = alpha_bars[t].unsqueeze(-1)
    x_t = ab_t * x_0 + (1 - ab_t) * eps

    eps_pred = model(x_t, t)

    loss = F.mse_loss(eps_pred, eps)

    # --- Bug 4: zero_grad after backward ---
    loss.backward()
    optimizer.zero_grad()
    optimizer.step()

# Images loaded as [0, 1] but not normalized to [-1, 1]
# transform = transforms.ToTensor()

# --- Sampling ---
def sample(model, n_samples=100):
    x = torch.randn(n_samples, 2)

    # --- Bug 6: Off-by-one, sampling includes t=0 noise addition ---
    for t_val in reversed(range(T)):  # goes T-1 ... 0
        t_batch = torch.full((n_samples,), t_val, dtype=torch.long)
        eps_pred = model(x, t_batch)

        beta_t = betas[t_val]
        alpha_t = alphas[t_val]

        # --- Bug 7: Using alpha_t instead of alpha_bar_t ---
        x = (1.0 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1.0 - alpha_t)) * eps_pred
        )

        # --- Bug 8: Adding noise at t=0 ---
        z = torch.randn_like(x)
        x = x + torch.sqrt(beta_t) * z

    return x

# model.eval() not called before sampling -- dropout/batchnorm behave differently

# eps_guided = eps_cond + w * (eps_cond - eps_uncond)
#   or equivalently: (1 - w) * eps_uncond + w * eps_cond
"""

print("Read the code above and find all 10 bugs before checking the solution below.")

# Answer -- Exercise 10.5: All 10 Bugs Explained

### Bug 1: `alpha_bars = alphas` (no cumulative product)
`alpha_bars` should be `torch.cumprod(alphas, dim=0)`, not a copy of `alphas`.

### Bug 2: Raw timestep instead of sinusoidal embedding
Feeding raw integer $t$ as a scalar gives the network almost no ability to distinguish nearby timesteps. Use a sinusoidal embedding that maps $t$ to a high-dimensional vector.

### Bug 3: Missing `sqrt` in forward process
Should be $x_t = \sqrt{\bar\alpha_t} x_0 + \sqrt{1-\bar\alpha_t} \epsilon$. Without square roots, the variance-preserving property breaks.

### Bug 4: `zero_grad()` after `backward()`
Calling `zero_grad()` after `backward()` erases the just-computed gradients, so `step()` updates with zeros. Must call `zero_grad()` **before** `backward()`.

### Bug 5: Images in [0, 1] instead of [-1, 1]
Diffusion assumes data centered near zero. Images in [0, 1] create a distribution mismatch at $t=T$ since sampling starts from $\mathcal{N}(0, 1)$.

### Bug 6: Off-by-one (combined with Bug 8)
The loop bounds `reversed(range(T))` are correct, but noise is added at $t=0$ (see Bug 8).

### Bug 7: Using `alpha_t` instead of `alpha_bar_t` in reverse mean
The DDPM reverse formula requires $\sqrt{1 - \bar\alpha_t}$ in the denominator, not $\sqrt{1 - \alpha_t}$.

### Bug 8: Adding noise at t=0
The final step ($t=0$) must be deterministic. Fix: `if t_val > 0: x = x + sqrt(beta_t) * z`

### Bug 9: Missing `model.eval()` during sampling
Without `eval()`, dropout stays active and BatchNorm uses batch statistics instead of running statistics.

### Bug 10: Wrong CFG formula
`eps_cond + w * (eps_cond - eps_uncond)` over-amplifies the conditional signal. Correct: `eps_uncond + w * (eps_cond - eps_uncond)`.

---

## Exercise 10.6: Design Discussion

**Prompt:** You have trained a diffusion model on 28x28 MNIST. Now you want to scale it to generate 512x512 photorealistic images. Walk through the major design decisions, architecture changes, and engineering considerations. Assume you have access to 8 A100 GPUs.

*Organize your answer into clear sections. Think about this for 10 minutes before reading the answer.*